# 故障予知② — 改善モデル：閾値調整・画像・不良種別の診断をテストで測る

出典：第05章 04節／パス：`05_inspection/02_improved.ipynb`／パッケージ：numpy・pandas・scikit-learn・matplotlib／実行時間の目安：約1分

このノート単体で実行できる。前半は `input/sensors.csv`（合成）の閾値調整（調整は調整用、評価はテストで1回）。
中盤は scikit-learn 同梱の数字画像（公開データ）で良品学習の置換え位置を示し、後半は UCI Steel Plates Faults（公開の実データ）で不良種別の診断を回す。
ベースラインのコードは `01_baseline.ipynb` に分けてある。検出器（標準化＋PCA＋再構成誤差）と分け方はベースラインと同じである。

In [1]:
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RS = 0


def make_sensor_data(n_normal: int = 5000, n_abnormal: int = 200, seed: int = 5):
    r = np.random.default_rng(seed)
    base = r.normal(0, 1, size=(n_normal, 8))
    normal = base + 0.6 * base[:, [0]]  # センサー間の相関
    shift = np.array([2.5, -2.0, 1.5, 0.5, 0.0, 0.0, 1.0, -1.0])
    abnormal = base[:n_abnormal] * 1.5 + shift + r.normal(0, 0.5, size=(n_abnormal, 8))
    X = np.vstack([normal, abnormal])
    y = np.concatenate([np.zeros(n_normal, dtype=int), np.ones(n_abnormal, dtype=int)])
    return X, y


try:
    _s = pd.read_csv("input/sensors.csv")
    print("Inputを使用: input/sensors.csv")
    X = _s[[f"s{i}" for i in range(8)]].to_numpy()
    y = _s["label"].to_numpy()
except FileNotFoundError:
    X, y = make_sensor_data()
    print("Inputが無いため合成データを生成")
print(f"件数: {len(y)}, 異常率: {y.mean():.4f}")
# 正常は学習用・閾値調整用（500件）・評価用に分ける。調整用はPCAの学習に使わない
Xn_rest, Xn_te = train_test_split(X[y == 0], test_size=0.25, random_state=RS)
Xn_tr, Xn_tune = train_test_split(Xn_rest, test_size=500, random_state=RS)
# 異常は閾値調整用（3割＝60件）と評価用（7割＝140件）に分ける。学習には使わない
Xa_tune, Xa_te = train_test_split(X[y == 1], test_size=0.70, random_state=RS)
print(f"正常 学習:{len(Xn_tr)} 調整:{len(Xn_tune)} 評価:{len(Xn_te)} / 異常 調整:{len(Xa_tune)} 評価:{len(Xa_te)}")

Inputを使用: input/sensors.csv
件数: 5200, 異常率: 0.0385
正常 学習:3250 調整:500 評価:1250 / 異常 調整:60 評価:140


In [2]:
# 正常だけを3次元に圧縮するPCAを学び、戻せない度合いを異常スコアにする
scaler = StandardScaler().fit(Xn_tr)
pca = PCA(n_components=3, random_state=RS).fit(scaler.transform(Xn_tr))


def anomaly_score(Z: np.ndarray) -> np.ndarray:
    Zs = scaler.transform(Z)
    return ((Zs - pca.inverse_transform(pca.transform(Zs))) ** 2).sum(axis=1)


X_te = np.vstack([Xn_te, Xa_te])
y_te = np.concatenate([np.zeros(len(Xn_te), dtype=int), np.ones(len(Xa_te), dtype=int)])
s_te = anomaly_score(X_te)
print(f"[再構成誤差] テストPR-AUC={average_precision_score(y_te, s_te):.4f}（異常率 {y_te.mean():.3f}）")

[再構成誤差] テストPR-AUC=0.8465（異常率 0.101）


In [3]:
# 改善：閾値を調整用（PCAの学習に使っていない正常500件＋異常60件）で決める
thr_fixed = float(np.quantile(anomaly_score(Xn_tr), 0.99))
s_tune_n = anomaly_score(Xn_tune)
s_tune_a = anomaly_score(Xa_tune)
# 選ぶ基準：点検能力から、正常の誤検知率を2%以内に抑え、その中で調整用の再現率を最大にする
MAX_FP_RATE = 0.02
print("閾値 / 調整用再現率 / 調整用誤検知率（正常）:")
best = None
for t in np.sort(s_tune_a):  # 異常の各スコアを閾値の候補にする
    rec, fpr = float((s_tune_a >= t).mean()), float((s_tune_n >= t).mean())
    if fpr <= MAX_FP_RATE and (best is None or rec > best[1]):
        best = (float(t), rec, fpr)
for t in [thr_fixed, best[0]]:
    print(f"{t:.2f} / {(s_tune_a >= t).mean():.3f} / {(s_tune_n >= t).mean():.3f}")
thr_tuned = best[0]
print(f"選んだ閾値: {thr_tuned:.2f}（誤検知率{MAX_FP_RATE:.0%}以内で再現率が最大）［調整用で選択］")


def counts(thr: float) -> tuple:
    """テストでの拾えた件数・見逃し・誤検知・点検量（旗を立てた件数）"""
    flag = s_te >= thr
    tp = int((flag & (y_te == 1)).sum())
    fp = int((flag & (y_te == 0)).sum())
    return tp, int((y_te == 1).sum()) - tp, fp, tp + fp


# テストで1回だけ測り、固定閾値と並べる（点検量と見逃しのトレードオフ）
print("閾値 / 再現率 / 適合率 / 拾えた / 見逃し / 誤検知 / 点検量［テスト］:")
rows = {}
for name, t in [("固定", thr_fixed), ("調整", thr_tuned)]:
    tp, fn, fp, n_flag = counts(t)
    rows[name] = (tp, fp)
    print(f"{name}{t:.2f} / {tp / (tp + fn):.3f} / {tp / n_flag:.3f} / {tp} / {fn} / {fp} / {n_flag}")
d_tp, d_fp = rows["調整"][0] - rows["固定"][0], rows["調整"][1] - rows["固定"][1]
print(f"[改善結果] 拾えた +{d_tp}件・誤検知 +{d_fp}件（見逃し1件を減らすのに誤検知 {d_fp / d_tp:.1f}件、テスト）")
assert np.isfinite(s_te).all()

閾値 / 調整用再現率 / 調整用誤検知率（正常）:
9.80 / 0.767 / 0.004
7.83 / 0.883 / 0.020
選んだ閾値: 7.83（誤検知率2%以内で再現率が最大）［調整用で選択］
閾値 / 再現率 / 適合率 / 拾えた / 見逃し / 誤検知 / 点検量［テスト］:
固定9.80 / 0.707 / 0.892 / 99 / 41 / 12 / 111
調整7.83 / 0.779 / 0.768 / 109 / 31 / 33 / 142
[改善結果] 拾えた +10件・誤検知 +21件（見逃し1件を減らすのに誤検知 2.1件、テスト）


In [4]:
# 画像の良品学習：数字「8」だけを正常として学び、他の数字を異常として検出する
from sklearn.datasets import load_digits

digits = load_digits()
X_img, y_img = digits.data / 16.0, digits.target  # 画素を0〜1にそろえる
is_ok = y_img == 8
print(f"画像: {len(y_img)}件（8x8ピクセル）/ 正常「8」: {is_ok.sum()}件")
X8_tr, X8_te = train_test_split(X_img[is_ok], test_size=0.5, random_state=RS)
# 実務の入れ替え位置：この X_img を学習済みCNNの特徴量に差し替える（学び方・評価の型は同じ）
pca_img = PCA(n_components=10, random_state=RS).fit(X8_tr)


def image_score(Z: np.ndarray) -> np.ndarray:
    return ((Z - pca_img.inverse_transform(pca_img.transform(Z))) ** 2).sum(axis=1)


s_ok, s_ng = image_score(X8_te), image_score(X_img[~is_ok])
# 異常を少数側（正常の約1割）に間引き、引き直しを20回して平均と幅を見る
n_ng = len(X8_te) // 10
r = np.random.default_rng(RS)
aps = []
for _ in range(20):
    pick = r.choice(len(s_ng), size=n_ng, replace=False)
    y_eval = np.concatenate([np.zeros(len(s_ok), dtype=int), np.ones(n_ng, dtype=int)])
    aps.append(average_precision_score(y_eval, np.concatenate([s_ok, s_ng[pick]])))
print(f"評価: 正常{len(X8_te)}件 / 異常{n_ng}件（異常率 {n_ng / (len(X8_te) + n_ng):.3f}）×20回")
print(f"[画像・再構成誤差] PR-AUC 平均={np.mean(aps):.3f}（最小 {np.min(aps):.3f}・最大 {np.max(aps):.3f}）")
assert np.isfinite(s_ng).all()

画像: 1797件（8x8ピクセル）/ 正常「8」: 174件
評価: 正常87件 / 異常8件（異常率 0.084）×20回
[画像・再構成誤差] PR-AUC 平均=0.867（最小 0.705・最大 0.970）


In [5]:
# 不良種別の診断の実演：鋼板の不良種別を当てる（UCI Steel Plates Faults、実データ1941件）
STEEL_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00198/Faults.NNA"
# 種別名は元データの綴りのまま（K_Scatch は Scratch の誤記だが変えない）
FAULTS = ["Pastry", "Z_Scratch", "K_Scatch", "Stains", "Dirtiness", "Bumps", "Other"]
STEEL_COLS = [f"F{i}" for i in range(1, 28)] + FAULTS


def load_steel() -> pd.DataFrame:
    df = pd.read_csv(STEEL_URL, sep="\t", header=None)
    df.columns = STEEL_COLS
    return df


try:
    st = load_steel()
    print("公開データを使用: UCI Steel Plates Faults（鋼板不良の実データ）")
except Exception:
    from sklearn.datasets import make_classification
    Xs, ys = make_classification(n_samples=1941, n_features=27, n_informative=15,
                                 n_classes=7, random_state=7)
    st = pd.DataFrame(Xs, columns=[f"F{i}" for i in range(1, 28)])
    for j, name in enumerate(FAULTS):
        st[name] = (ys == j).astype(int)
    print("取得に失敗したため合成データを使用（手順確認用）")
st["fault"] = st[FAULTS].to_numpy().argmax(axis=1)
print(f"件数: {len(st)}, 不良種別ごとの件数:")
for k, name in enumerate(FAULTS):
    print(f"  {name}: {(st['fault'] == k).sum()}")

公開データを使用: UCI Steel Plates Faults（鋼板不良の実データ）
件数: 1941, 不良種別ごとの件数:
  Pastry: 158
  Z_Scratch: 190
  K_Scatch: 391
  Stains: 72
  Dirtiness: 55
  Bumps: 402
  Other: 673


In [6]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score

Xf = st[[f"F{i}" for i in range(1, 28)]]
yf = st["fault"].to_numpy()
Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(Xf, yf, test_size=0.25, stratify=yf, random_state=RS)
clf = HistGradientBoostingClassifier(max_iter=200, random_state=RS).fit(Xf_tr, yf_tr)
pf = clf.predict(Xf_te)
print(f"[不良種別の診断] テスト正解率={accuracy_score(yf_te, pf):.3f}")
print("種別ごとの再現率:")
for k, name in enumerate(FAULTS):
    m = yf_te == k
    print(f"  {name}: {((pf[m] == k).sum() / m.sum()):.3f}（{m.sum()}件）")
assert np.isfinite(Xf_te.to_numpy()).all()

[不良種別の診断] テスト正解率=0.817
種別ごとの再現率:
  Pastry: 0.590（39件）
  Z_Scratch: 0.917（48件）
  K_Scatch: 0.949（98件）
  Stains: 0.833（18件）
  Dirtiness: 0.786（14件）
  Bumps: 0.752（101件）
  Other: 0.804（168件）


In [7]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
_book_fig = Path("../../../books/python-machine-jitumu/assets/images")
FIG = str(_book_fig if _book_fig.is_dir() else Path("outputs"))
Path(FIG).mkdir(parents=True, exist_ok=True)
plt.figure()
plt.hist(s_te[y_te == 0], bins=40, alpha=0.6, label="normal")
plt.hist(s_te[y_te == 1], bins=20, alpha=0.6, label="anomaly")
plt.axvline(thr_fixed, color="black", linestyle="--", label=f"fixed threshold ({thr_fixed:.2f})")
plt.axvline(thr_tuned, color="red", linestyle="--", label=f"tuned threshold ({thr_tuned:.2f})")
plt.xlabel("reconstruction error"); plt.ylabel("count"); plt.legend(); plt.title("score distributions (test, improved)")
plt.savefig(f"{FIG}/code-result-j05-02.png", dpi=150, bbox_inches="tight"); plt.close()
print("saved code-result-j05-02.png")

saved code-result-j05-02.png


In [8]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
ng_all = X_img[~is_ok]
for j in range(5):
    axes[0, j].imshow(X8_te[j].reshape(8, 8), cmap="gray")
    axes[0, j].set_title("normal"); axes[0, j].axis("off")
    axes[1, j].imshow(ng_all[j * 7].reshape(8, 8), cmap="gray")
    axes[1, j].set_title("anomaly"); axes[1, j].axis("off")
fig.suptitle("digits: normal (8) vs anomalies")
fig.savefig(f"{FIG}/code-result-j05-03.png", dpi=150, bbox_inches="tight"); plt.close(fig)
print("saved code-result-j05-03.png")

saved code-result-j05-03.png


## 読み取りどころ

- 閾値は「点検能力（誤検知率2%以内）」という基準で調整用から選び、テストで1回だけ測ること。
- 改善は「拾えた件数の増加」と「誤検知の増加」の交換であり、見逃し1件の損失と点検の手間で採否を決めること。
- 画像では画素をCNN特徴に差し替える位置だけが変わり、学び方・評価の型は同じであること。異常が少ないと評価はぶれること。
- 不良種別の診断は種別ごとの再現率で苦手を見つけること。